# Kaggle Submission: exp_007 Native 3-Fold Hybrid

This notebook is intended for direct upload into the BirdCLEF+ 2026 Kaggle competition.

What you need before running it:
- Attach the competition data.
- Attach a Kaggle Dataset that contains the `exp_006` fold checkpoints `fold_00/best_model.pt`, `fold_01/best_model.pt`, and `fold_02/best_model.pt`.
- Keep internet disabled.

What it does:
- Finds the competition data and all `exp_006` fold checkpoints.
- Reconstructs the repository-native soundscape finetuning model.
- Builds fold-specific `site/hour/site-hour` priors from labeled soundscapes.
- Runs a 3-fold ensemble with the validated `exp_007` recipe: `event + texture priors + smoothing`.
- Writes `/kaggle/working/submission.csv`.


In [ ]:
import os
import re
import time
import warnings
from contextlib import nullcontext
from dataclasses import dataclass
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchaudio.transforms as T

warnings.filterwarnings("ignore")
START = time.time()

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

if device.type == "cpu":
    torch.set_num_threads(max(1, os.cpu_count() or 1))

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

print("Torch:", torch.__version__)
print("Device:", device)
if device.type == "cpu":
    print("CPU threads:", torch.get_num_threads())


## Configuration

- `MODEL_DATASET_HINT` is optional. Set it only if you know the dataset folder name under `/kaggle/input/` that contains the `exp_006` fold checkpoints.
- `FOLD_IDS` defaults to `(0, 1, 2)` and should match the uploaded fold directories.
- The inference recipe is intentionally fixed to the best local `exp_007` setting.
- The notebook can fall back to aligned `train_soundscapes` for a dry run when hidden test audio is not mounted yet.


In [ ]:
MODEL_DATASET_HINT = None  # Example: "birdclef-2026-exp006-folds"
CHECKPOINT_NAME = "best_model.pt"
FOLD_IDS = (0, 1, 2)
LAMBDA_EVENT = 0.4
LAMBDA_TEXTURE = 1.0
SMOOTH_TEXTURE = 0.35
TEXTURE_TAXA = {"Amphibia", "Insecta"}
AUDIO_SUFFIXES = {".ogg", ".wav", ".flac", ".mp3"}


@dataclass
class Config:
    sr: int = 32_000
    chunk_duration: float = 5.0
    n_mels: int = 224
    n_fft: int = 2048
    hop_length: int = 512
    fmin: int = 0
    fmax: int = 16_000
    top_db: float = 80.0
    power: float = 2.0
    norm: str = "slaney"
    mel_scale: str = "htk"
    backbone: str = "tf_efficientnet_b0.ns_jft_in1k"
    num_classes: int = 234
    in_channels: int = 3
    dropout: float = 0.1
    drop_path_rate: float = 0.0
    gem_p_init: float = 3.0

    @property
    def chunk_samples(self) -> int:
        return int(self.sr * self.chunk_duration)


def resolve_data_root() -> Path:
    input_root = Path("/kaggle/input")
    direct_candidates = [
        input_root / "competitions" / "birdclef-2026",
        input_root / "birdclef-2026",
    ]
    for candidate in direct_candidates:
        if (candidate / "sample_submission.csv").exists():
            return candidate

    for child in sorted(input_root.iterdir()):
        if (child / "sample_submission.csv").exists():
            return child
        matches = sorted(child.rglob("sample_submission.csv"))
        if matches:
            return matches[0].parent

    raise FileNotFoundError("BirdCLEF competition data was not found under /kaggle/input.")


def iter_candidate_roots(model_dataset_hint: str | None):
    input_root = Path("/kaggle/input")

    if model_dataset_hint:
        hinted_path = Path(model_dataset_hint)
        if hinted_path.exists():
            yield hinted_path
        else:
            hinted_under_input = input_root / model_dataset_hint
            if hinted_under_input.exists():
                yield hinted_under_input

    for child in sorted(input_root.iterdir()):
        yield child


def resolve_fold_checkpoint_paths(model_dataset_hint: str | None = None, checkpoint_name: str = "best_model.pt", fold_ids=FOLD_IDS) -> dict[int, Path]:
    searched = []
    seen = set()
    found = {}

    for root in iter_candidate_roots(model_dataset_hint):
        try:
            root = root.resolve()
        except FileNotFoundError:
            continue

        root_key = str(root)
        if root_key in seen:
            continue
        seen.add(root_key)
        searched.append(root_key)

        if root.is_dir():
            for path in sorted(root.rglob(checkpoint_name)):
                fold = None
                for part in path.parts:
                    if re.fullmatch(r"fold_\d{2}", part):
                        fold = int(part.split("_")[1])
                        break
                if fold in fold_ids and fold not in found:
                    found[fold] = path
        elif root.is_file() and root.name == checkpoint_name:
            for part in root.parts:
                if re.fullmatch(r"fold_\d{2}", part):
                    fold = int(part.split("_")[1])
                    if fold in fold_ids and fold not in found:
                        found[fold] = root
                    break

        if len(found) == len(tuple(fold_ids)):
            break

    missing = [int(fold) for fold in fold_ids if int(fold) not in found]
    if missing:
        raise FileNotFoundError(
            f"Could not find fold checkpoints for folds {missing} under /kaggle/input. Searched roots: {searched[:10]}"
        )
    return {int(fold): found[int(fold)] for fold in fold_ids}


row_pattern = re.compile(r"^(.*)_(\d+)$")


def parse_row_id(row_id: str):
    match = row_pattern.match(str(row_id))
    if not match:
        return None, None
    return match.group(1), int(match.group(2))


def parse_soundscape_key(name_or_stem: str):
    stem = Path(name_or_stem).stem
    parts = stem.split("_")
    site = parts[3] if len(parts) >= 4 else None
    time_token = parts[-1] if parts else "000000"
    hour_utc = int(time_token[:2]) if time_token.isdigit() and len(time_token) >= 2 else -1
    return site, hour_utc


cfg = Config()
DATA_ROOT = resolve_data_root()
CHECKPOINT_PATHS = resolve_fold_checkpoint_paths(MODEL_DATASET_HINT, CHECKPOINT_NAME, FOLD_IDS)
SAMPLE_SUB = pd.read_csv(DATA_ROOT / "sample_submission.csv")
SPECIES = SAMPLE_SUB.columns[1:].tolist()
TEST_DIR = DATA_ROOT / "test_soundscapes"
TRAIN_SC_DIR = DATA_ROOT / "train_soundscapes"

print({
    "data_root": str(DATA_ROOT),
    "fold_ids": [int(fold) for fold in FOLD_IDS],
    "checkpoint_paths": {int(k): str(v) for k, v in CHECKPOINT_PATHS.items()},
    "num_classes": len(SPECIES),
    "lambda_event": LAMBDA_EVENT,
    "lambda_texture": LAMBDA_TEXTURE,
    "smooth_texture": SMOOTH_TEXTURE,
    "device": str(device),
})


## Model Blocks

The architecture below matches the checkpoints used by `exp_006_soundscape_finetuning_v2`.

Key details:
- each fold checkpoint stores the full `WaveformSEDClassifier` state dict
- this includes the mel frontend inside the wrapper
- inference therefore runs directly on waveform chunks
- the final prediction is the mean of fold-specific fused logits


In [ ]:
@dataclass
class ReferenceModelConfig:
    sr: int = 32_000
    chunk_duration: float = 5.0
    n_mels: int = 224
    n_fft: int = 2048
    hop_length: int = 512
    fmin: int = 0
    fmax: int = 16_000
    top_db: float = 80.0
    power: float = 2.0
    norm: str = "slaney"
    mel_scale: str = "htk"
    backbone: str = "tf_efficientnet_b0.ns_jft_in1k"
    num_classes: int = 234
    in_channels: int = 3
    dropout: float = 0.1
    drop_path_rate: float = 0.0
    gem_p_init: float = 3.0


class GEMFreqPool(nn.Module):
    def __init__(self, p_init: float = 3.0, eps: float = 1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init))
        self.eps = eps

    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)


class AttentionSEDHead(nn.Module):
    def __init__(self, feat_dim: int, num_classes: int, dropout: float = 0.1):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.att_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
        self.cls_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.fc(x)
        x = x.permute(0, 2, 1)
        att = torch.tanh(self.att_conv(x))
        att = F.softmax(att, dim=-1)
        cls = self.cls_conv(x)
        clipwise_logit = (att * cls).sum(dim=-1)
        clipwise_prob = torch.sigmoid(clipwise_logit)
        return {
            "clipwise_logit": clipwise_logit,
            "clipwise_prob": clipwise_prob,
        }


class SEDModel(nn.Module):
    def __init__(self, cfg: ReferenceModelConfig):
        super().__init__()
        self.backbone = timm.create_model(
            cfg.backbone,
            pretrained=False,
            in_chans=cfg.in_channels,
            features_only=False,
            global_pool="",
            num_classes=0,
            drop_path_rate=cfg.drop_path_rate,
        )
        feat_dim = self.backbone.num_features
        self.gem_pool = GEMFreqPool(p_init=cfg.gem_p_init)
        self.head = AttentionSEDHead(feat_dim, cfg.num_classes, cfg.dropout)

    def forward(self, x):
        features = self.backbone(x)
        pooled = self.gem_pool(features)
        return self.head(pooled)


class MelSpectrogramTransform(nn.Module):
    def __init__(self, cfg: ReferenceModelConfig):
        super().__init__()
        self.mel = T.MelSpectrogram(
            sample_rate=cfg.sr,
            n_fft=cfg.n_fft,
            hop_length=cfg.hop_length,
            n_mels=cfg.n_mels,
            f_min=cfg.fmin,
            f_max=cfg.fmax,
            power=cfg.power,
            norm=cfg.norm,
            mel_scale=cfg.mel_scale,
        )
        self.db = T.AmplitudeToDB(stype="power", top_db=cfg.top_db)

    @torch.no_grad()
    def forward(self, waveforms):
        waveforms = torch.nan_to_num(waveforms.float(), nan=0.0, posinf=0.0, neginf=0.0)
        mel = self.mel(waveforms)
        mel = torch.nan_to_num(mel, nan=0.0, posinf=0.0, neginf=0.0)
        mel = self.db(mel)
        mel = torch.nan_to_num(mel, nan=-80.0, posinf=0.0, neginf=-80.0)

        batch_size = mel.shape[0]
        mel_flat = mel.reshape(batch_size, -1)
        mel_min = mel_flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel_max = mel_flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel = (mel - mel_min) / (mel_max - mel_min + 1e-7)
        mel = torch.nan_to_num(mel, nan=0.0, posinf=1.0, neginf=0.0)
        return mel.unsqueeze(1).repeat(1, 3, 1, 1)


class WaveformSEDClassifier(nn.Module):
    def __init__(self, cfg: ReferenceModelConfig):
        super().__init__()
        self.mel = MelSpectrogramTransform(cfg)
        self.model = SEDModel(cfg)

    def forward(self, waveforms):
        mel = self.mel(waveforms)
        return self.model(mel)


def safe_load_checkpoint(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def extract_state_dict(ckpt):
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        return ckpt["model_state_dict"]
    return ckpt


def extract_checkpoint_meta(ckpt):
    if not isinstance(ckpt, dict):
        return {"stage": "raw_state_dict"}
    metrics = ckpt.get("metrics", {})
    return {
        "epoch": ckpt.get("epoch"),
        "stage": ckpt.get("stage"),
        "metrics": metrics if isinstance(metrics, dict) else {},
    }


def load_model(ckpt_path: Path, cfg_native: Config):
    model_cfg = ReferenceModelConfig(num_classes=cfg_native.num_classes)
    model = WaveformSEDClassifier(model_cfg)
    ckpt = safe_load_checkpoint(ckpt_path)
    model.load_state_dict(extract_state_dict(ckpt), strict=True)
    model.to(device).eval()
    return model, extract_checkpoint_meta(ckpt)


def sigmoid(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32, copy=False)
    positive = x >= 0
    out = np.empty_like(x, dtype=np.float32)
    out[positive] = 1.0 / (1.0 + np.exp(-x[positive]))
    exp_x = np.exp(x[~positive])
    out[~positive] = exp_x / (1.0 + exp_x)
    return out


ensemble = []
checkpoint_meta = {}
for fold in FOLD_IDS:
    model, meta = load_model(CHECKPOINT_PATHS[int(fold)], cfg)
    ensemble.append((int(fold), model))
    checkpoint_meta[int(fold)] = meta

print("Loaded fold checkpoints:")
for fold in FOLD_IDS:
    print(f"  fold_{int(fold):02d}:", CHECKPOINT_PATHS[int(fold)].name, checkpoint_meta[int(fold)])


## Fit Fold-Specific Metadata Priors

This cell uses only the public training-side soundscape labels.

The resulting prior tables mirror the validated `exp_007` setup:
- priors are built separately for each fold's non-held-out partition
- global, site, hour, and site-hour tables are all available
- texture classes are identified from `taxonomy.csv` as `Amphibia` and `Insecta`


In [ ]:
def parse_time_to_seconds(value):
    text = str(value)
    parts = text.split(":")
    if len(parts) == 3:
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(float(parts[2]))
    if len(parts) == 2:
        return int(parts[0]) * 60 + int(float(parts[1]))
    return int(float(text))


def parse_soundscape_labels(value):
    if pd.isna(value):
        return []
    return [token.strip() for token in str(value).split(";") if token.strip()]


def union_labels(series: pd.Series):
    merged = []
    seen = set()
    for value in series:
        for label in parse_soundscape_labels(value):
            if label not in seen:
                merged.append(label)
                seen.add(label)
    return merged


def encode_multi_hot(labels, species, label_to_idx):
    target = np.zeros(len(species), dtype=np.float32)
    for label in labels:
        idx = label_to_idx.get(label)
        if idx is not None:
            target[idx] = 1.0
    return target


def fit_prior_tables(prior_df: pd.DataFrame, y_prior: np.ndarray):
    prior_df = prior_df.reset_index(drop=True)
    global_p = y_prior.mean(axis=0).astype(np.float32)

    site_keys = sorted(prior_df["site"].dropna().astype(str).unique().tolist())
    site_to_i = {key: idx for idx, key in enumerate(site_keys)}
    site_n = np.zeros(len(site_keys), dtype=np.float32)
    site_p = np.zeros((len(site_keys), y_prior.shape[1]), dtype=np.float32)
    site_series = prior_df["site"].astype(str)
    for site in site_keys:
        idx = site_to_i[site]
        mask = site_series.values == site
        site_n[idx] = mask.sum()
        site_p[idx] = y_prior[mask].mean(axis=0)

    hour_keys = sorted(prior_df["hour_utc"].dropna().astype(int).unique().tolist())
    hour_to_i = {hour: idx for idx, hour in enumerate(hour_keys)}
    hour_n = np.zeros(len(hour_keys), dtype=np.float32)
    hour_p = np.zeros((len(hour_keys), y_prior.shape[1]), dtype=np.float32)
    hour_series = prior_df["hour_utc"].astype(int)
    for hour in hour_keys:
        idx = hour_to_i[hour]
        mask = hour_series.values == hour
        hour_n[idx] = mask.sum()
        hour_p[idx] = y_prior[mask].mean(axis=0)

    sh_to_i = {}
    sh_n_list = []
    sh_p_list = []
    grouped = prior_df.groupby(["site", "hour_utc"], dropna=False).groups
    for (site, hour), group_idx in grouped.items():
        site_key = str(site) if pd.notna(site) else None
        hour_key = int(hour) if pd.notna(hour) else -1
        if site_key is None or hour_key < 0:
            continue
        sh_to_i[(site_key, hour_key)] = len(sh_n_list)
        group_idx = np.asarray(list(group_idx), dtype=np.int64)
        sh_n_list.append(len(group_idx))
        sh_p_list.append(y_prior[group_idx].mean(axis=0))

    if sh_p_list:
        sh_p = np.stack(sh_p_list).astype(np.float32)
        sh_n = np.asarray(sh_n_list, dtype=np.float32)
    else:
        sh_p = np.zeros((0, y_prior.shape[1]), dtype=np.float32)
        sh_n = np.zeros((0,), dtype=np.float32)

    return {
        "global_p": global_p,
        "site_to_i": site_to_i,
        "site_n": site_n,
        "site_p": site_p,
        "hour_to_i": hour_to_i,
        "hour_n": hour_n,
        "hour_p": hour_p,
        "sh_to_i": sh_to_i,
        "sh_n": sh_n,
        "sh_p": sh_p,
    }


def prior_logits_from_tables(sites, hours, tables, eps=1e-4):
    n_rows = len(sites)
    p = np.repeat(tables["global_p"][None, :], n_rows, axis=0).astype(np.float32, copy=True)

    site_idx = np.fromiter((tables["site_to_i"].get(str(site), -1) for site in sites), dtype=np.int32, count=n_rows)
    hour_idx = np.fromiter((tables["hour_to_i"].get(int(hour), -1) if int(hour) >= 0 else -1 for hour in hours), dtype=np.int32, count=n_rows)
    sh_idx = np.fromiter(
        (tables["sh_to_i"].get((str(site), int(hour)), -1) if int(hour) >= 0 else -1 for site, hour in zip(sites, hours)),
        dtype=np.int32,
        count=n_rows,
    )

    valid = hour_idx >= 0
    if valid.any():
        nh = tables["hour_n"][hour_idx[valid]][:, None]
        wh = nh / (nh + 8.0)
        p[valid] = wh * tables["hour_p"][hour_idx[valid]] + (1.0 - wh) * p[valid]

    valid = site_idx >= 0
    if valid.any():
        ns = tables["site_n"][site_idx[valid]][:, None]
        ws = ns / (ns + 8.0)
        p[valid] = ws * tables["site_p"][site_idx[valid]] + (1.0 - ws) * p[valid]

    valid = sh_idx >= 0
    if valid.any():
        nsh = tables["sh_n"][sh_idx[valid]][:, None]
        wsh = nsh / (nsh + 4.0)
        p[valid] = wsh * tables["sh_p"][sh_idx[valid]] + (1.0 - wsh) * p[valid]

    np.clip(p, eps, 1.0 - eps, out=p)
    return (np.log(p) - np.log1p(-p)).astype(np.float32, copy=False)


def smooth_sequence_cols(scores: np.ndarray, cols: np.ndarray, alpha: float = 0.35) -> np.ndarray:
    if alpha <= 0 or len(cols) == 0 or len(scores) <= 1:
        return scores.copy()
    smoothed = scores.copy()
    x = smoothed[:, cols]
    prev_x = np.concatenate([x[:1], x[:-1]], axis=0)
    next_x = np.concatenate([x[1:], x[-1:]], axis=0)
    smoothed[:, cols] = (1.0 - alpha) * x + 0.5 * alpha * (prev_x + next_x)
    return smoothed


def fuse_native_logits(logits: np.ndarray, prior_logits: np.ndarray, idx_event: np.ndarray, idx_texture: np.ndarray):
    fused = logits.copy()
    if len(idx_event):
        fused[:, idx_event] += LAMBDA_EVENT * prior_logits[:, idx_event]
    if len(idx_texture):
        fused[:, idx_texture] += LAMBDA_TEXTURE * prior_logits[:, idx_texture]
        fused = smooth_sequence_cols(fused, idx_texture, alpha=SMOOTH_TEXTURE)
    return fused


soundscape_labels = pd.read_csv(DATA_ROOT / "train_soundscapes_labels.csv")
taxonomy = pd.read_csv(DATA_ROOT / "taxonomy.csv")
label_to_idx = {label: idx for idx, label in enumerate(SPECIES)}

sc_clean = (
    soundscape_labels
    .groupby(["filename", "start", "end"])["primary_label"]
    .apply(union_labels)
    .reset_index(name="label_list")
)
sc_clean["start_sec"] = sc_clean["start"].map(parse_time_to_seconds).astype(int)
sc_clean["end_sec"] = sc_clean["end"].map(parse_time_to_seconds).astype(int)
sc_clean["row_id"] = sc_clean["filename"].str.replace(".ogg", "", regex=False) + "_" + sc_clean["end_sec"].astype(str)
sc_clean["site"], sc_clean["hour_utc"] = zip(*sc_clean["filename"].map(parse_soundscape_key))
sc_clean = sc_clean.sort_values(["filename", "end_sec"]).reset_index(drop=True)

windows_per_file = sc_clean.groupby("filename").size()
full_files = sorted(windows_per_file[windows_per_file == 12].index.tolist())
full_df = sc_clean[sc_clean["filename"].isin(full_files)].copy().reset_index(drop=True)

n_group_splits = min(5, max(2, len(full_files)))
gkf = GroupKFold(n_splits=n_group_splits)
full_splits = list(gkf.split(full_df, groups=full_df["filename"]))

texture_labels = taxonomy.loc[taxonomy["class_name"].isin(TEXTURE_TAXA), "primary_label"].tolist()
texture_set = set(texture_labels)
idx_texture = np.array([label_to_idx[label] for label in SPECIES if label in texture_set], dtype=np.int32)
idx_event = np.array([idx for idx, label in enumerate(SPECIES) if label not in texture_set], dtype=np.int32)

fold_prior_tables = {}
fold_train_counts = {}
for fold in FOLD_IDS:
    train_idx, valid_idx = full_splits[int(fold)]
    valid_files = set(full_df.iloc[valid_idx]["filename"].tolist())
    prior_df = (
        sc_clean[~sc_clean["filename"].isin(valid_files)]
        .sort_values(["filename", "end_sec"])
        .reset_index(drop=True)
    )
    y_prior = np.stack([encode_multi_hot(labels, SPECIES, label_to_idx) for labels in prior_df["label_list"]], axis=0)
    fold_prior_tables[int(fold)] = fit_prior_tables(prior_df[["site", "hour_utc"]], y_prior)
    fold_train_counts[int(fold)] = int(len(prior_df))

print({
    "labeled_segments": int(len(sc_clean)),
    "fully_labeled_files": int(len(full_files)),
    "texture_classes": int(len(idx_texture)),
    "event_classes": int(len(idx_event)),
    "fold_train_segments": fold_train_counts,
    "site_hour_priors_per_fold": {int(fold): int(len(fold_prior_tables[int(fold)]["sh_to_i"])) for fold in FOLD_IDS},
})


## Build Submission Row Mapping

The notebook reconstructs the expected output order from `sample_submission.csv` and then tries to find matching test audio files.

If hidden test audio is not mounted yet, the notebook falls back to aligned `train_soundscapes` so you can still do a dry run and verify that `submission.csv` is produced.


In [ ]:
expected_ids = SAMPLE_SUB["row_id"].tolist()
expected_by_stem = {}
for row_id in expected_ids:
    stem, end_sec = parse_row_id(row_id)
    if stem is None:
        continue
    expected_by_stem.setdefault(stem, []).append(end_sec)

for stem in expected_by_stem:
    expected_by_stem[stem] = sorted(expected_by_stem[stem])


def discover_test_files(data_root: Path, expected_stems: set[str]):
    search_roots = []
    for name in ["test_soundscapes", "test_audio"]:
        candidate = data_root / name
        if candidate.exists():
            search_roots.append(candidate)
    search_roots.append(data_root)

    matches = {}
    for root in search_roots:
        if not root.exists():
            continue
        for path in root.rglob("*"):
            if not path.is_file() or path.suffix.lower() not in AUDIO_SUFFIXES:
                continue
            stem = path.stem
            if stem in expected_stems and stem not in matches:
                matches[stem] = path
        if len(matches) == len(expected_stems):
            break

    return [matches[stem] for stem in sorted(matches)]


expected_stems = set(expected_by_stem)
test_files = discover_test_files(DATA_ROOT, expected_stems)
used_test_fallback = False

if len(test_files) == 0:
    fallback_files = []
    for stem in sorted(expected_stems):
        for suffix in sorted(AUDIO_SUFFIXES):
            candidate = TRAIN_SC_DIR / f"{stem}{suffix}"
            if candidate.exists():
                fallback_files.append(candidate)
                break
    test_files = fallback_files
    used_test_fallback = True
    print("Warning: hidden test files not found; using aligned train fallback for a dry run.")

matched_stems = {path.stem for path in test_files}
missing_stems = sorted(expected_stems - matched_stems)

print({
    "competition_dir": str(DATA_ROOT),
    "used_test_fallback": used_test_fallback,
    "matched_test_stems": len(matched_stems),
    "expected_test_stems": len(expected_stems),
    "missing_test_stems": len(missing_stems),
})
if test_files:
    print("First test path:", test_files[0])
if missing_stems:
    print("Missing test stems sample:", missing_stems[:5])


## Inference And Submission

This is the native 3-fold hybrid inference path:
- `exp_006` fold checkpoints `0`, `1`, `2`
- no external blend
- no Perch
- fold-specific `exp_007` metadata priors
- texture-aware smoothing
- final mean over fused fold logits


In [ ]:
def load_soundscape(path: Path) -> np.ndarray:
    audio, _ = librosa.load(path, sr=cfg.sr, mono=True)
    return np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def build_submission(row_ids, preds, expected_ids, species):
    if len(preds) == 0:
        pred_df = pd.DataFrame(
            np.zeros((0, len(species)), dtype=np.float32),
            columns=species,
            index=pd.Index([], name="row_id"),
        )
    else:
        pred_df = pd.DataFrame(
            np.asarray(preds, dtype=np.float32),
            columns=species,
            index=pd.Index(row_ids, name="row_id"),
        )

    pred_df = pred_df[~pred_df.index.duplicated(keep="first")]

    missing_ids = [row_id for row_id in expected_ids if row_id not in pred_df.index]
    if missing_ids:
        zeros = pd.DataFrame(
            np.zeros((len(missing_ids), len(species)), dtype=np.float32),
            columns=species,
            index=pd.Index(missing_ids, name="row_id"),
        )
        pred_df = pd.concat([pred_df, zeros], axis=0)

    pred_df = pred_df.loc[expected_ids]
    return pred_df.reset_index(), len(missing_ids)


CHUNK = cfg.chunk_samples
amp_enabled = device.type == "cuda"
all_row_ids = []
all_preds = []
processed_files = 0

print("Running exp_007 native 3-fold hybrid inference...")
with torch.no_grad():
    for file_idx, path in enumerate(test_files, start=1):
        stem = path.stem
        audio = load_soundscape(path)
        expected_secs = expected_by_stem.get(stem, [])

        if expected_secs:
            n_chunks = max(1, max(expected_secs) // int(cfg.chunk_duration))
        else:
            n_chunks = max(1, int(np.ceil(len(audio) / CHUNK)))

        padded_len = n_chunks * CHUNK
        if len(audio) < padded_len:
            audio = np.pad(audio, (0, padded_len - len(audio)))
        else:
            audio = audio[:padded_len]

        chunks = audio.reshape(n_chunks, CHUNK)
        chunk_tensor = torch.from_numpy(chunks).float().to(device)

        if expected_secs:
            valid_indices = [(end_sec // int(cfg.chunk_duration)) - 1 for end_sec in expected_secs]
            valid_indices = [idx for idx in valid_indices if 0 <= idx < n_chunks]
            if not valid_indices:
                continue
            target_row_ids = [f"{stem}_{(idx + 1) * int(cfg.chunk_duration)}" for idx in valid_indices]
        else:
            valid_indices = list(range(n_chunks))
            target_row_ids = [f"{stem}_{(idx + 1) * int(cfg.chunk_duration)}" for idx in valid_indices]

        site, hour_utc = parse_soundscape_key(stem)
        fused_logits_per_fold = []

        for fold, model in ensemble:
            with (torch.amp.autocast(device_type="cuda", enabled=amp_enabled) if amp_enabled else nullcontext()):
                output = model(chunk_tensor)
            logits = output["clipwise_logit"].detach().cpu().numpy().astype(np.float32)
            selected_logits = logits[valid_indices]

            prior_logits = prior_logits_from_tables(
                sites=np.array([site] * len(target_row_ids), dtype=object),
                hours=np.full(len(target_row_ids), hour_utc, dtype=np.int32),
                tables=fold_prior_tables[int(fold)],
            )
            fused_logits = fuse_native_logits(selected_logits, prior_logits, idx_event, idx_texture)
            fused_logits_per_fold.append(fused_logits)

        ensemble_logits = np.mean(np.stack(fused_logits_per_fold, axis=0), axis=0)
        fused_probs = sigmoid(ensemble_logits)

        all_row_ids.extend(target_row_ids)
        all_preds.extend(fused_probs)
        processed_files += 1

        if file_idx % 10 == 0 or file_idx == len(test_files):
            print(f"Processed {file_idx}/{len(test_files)} files")

submission, missing_count = build_submission(all_row_ids, all_preds, expected_ids, SPECIES)
submission.to_csv("/kaggle/working/submission.csv", index=False)

print("Missing row_ids filled with zeros:", missing_count)
print("Submission shape:", submission.shape)
print("Saved:", "/kaggle/working/submission.csv")
print("Elapsed minutes:", round((time.time() - START) / 60.0, 2))
submission.head()


## What To Do On Kaggle

1. Upload this notebook to the BirdCLEF+ 2026 competition.
2. Attach the competition data.
3. Attach your checkpoint dataset containing `fold_00/best_model.pt`, `fold_01/best_model.pt`, and `fold_02/best_model.pt` from `exp_006_soundscape_finetuning_v2`.
4. If your checkpoint dataset preserves the original folder tree, that is fine; the notebook searches recursively.
5. Keep `Internet` disabled.
6. Run `Save Version` and choose `Save & Submit`.
7. Copy the public LB score back into the project notes so we can compare it against the current native public result `0.737`.
